# Household Power Consumption — Binary Classification (High / Low)

**Objective**: Predict whether a minute's power consumption is **High** or **Low**.

**Approach**: Use the **raw minute-level sensor readings** (no PCA, no feature engineering), and compare multiple classifiers:
- Logistic Regression
- Random Forest
- XGBoost
- LightGBM
- Feed-Forward Neural Network (PyTorch)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_auc_score, roc_curve, f1_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## 2. Load Data

In [ ]:
df = pd.read_csv('../data/household_power_consumption.txt', sep=';',
                 low_memory=False, na_values=['?'])

num_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage',
            'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()

## 3. Handle Missing Values

In [ ]:
print(f"Rows before: {len(df):,}")
df.dropna(subset=num_cols, inplace=True)
df.drop(columns=['Date', 'Time'], inplace=True)
print(f"Rows after:  {len(df):,}")

## 4. Create Binary Target

Split on the **median** of `Global_active_power`.
- **0 = Low** (below or equal to median)
- **1 = High** (above median)

In [ ]:
threshold = df['Global_active_power'].median()
df['Label'] = (df['Global_active_power'] > threshold).astype(int)

print(f"Threshold (median): {threshold:.4f} kW")
print(f"\nClass counts:\n{df['Label'].value_counts().rename({0:'Low',1:'High'})}")
print(f"Class balance: {df['Label'].mean():.2%} High")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['Global_active_power'], bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Median = {threshold:.2f}')
axes[0].set_xlabel('Global Active Power (kW)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Minute-Level Power')
axes[0].legend()

df['Label'].value_counts().plot.bar(ax=axes[1], color=['#4a90d9', '#e74c3c'], edgecolor='white')
axes[1].set_xticklabels(['Low (0)', 'High (1)'], rotation=0)
axes[1].set_title('Class Distribution')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. Quick EDA

In [ ]:
plt.figure(figsize=(8, 6))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlations')
plt.tight_layout()
plt.show()

## 6. Prepare Features & Split

We use the **6 raw sensor features** (excluding `Global_active_power` which defines the target).

In [ ]:
feature_cols = ['Global_reactive_power', 'Voltage', 'Global_intensity',
                'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

X = df[feature_cols].values
y = df['Label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Features: {feature_cols}")
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

## 7. Train Models

### 7.1 Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_preds = lr.predict(X_test_sc)
lr_probs = lr.predict_proba(X_test_sc)[:, 1]
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, lr_preds):.4f}")

### 7.2 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]
print(f"Random Forest Accuracy: {accuracy_score(y_test, rf_preds):.4f}")

### 7.3 XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42, verbosity=0,
    tree_method='gpu_hist' if torch.cuda.is_available() else 'hist'
)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
xgb_probs = xgb.predict_proba(X_test)[:, 1]
print(f"XGBoost Accuracy: {accuracy_score(y_test, xgb_preds):.4f}")

### 7.4 LightGBM

In [ ]:
lgbm = LGBMClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbose=-1
)
lgbm.fit(X_train, y_train)
lgbm_preds = lgbm.predict(X_test)
lgbm_probs = lgbm.predict_proba(X_test)[:, 1]
print(f"LightGBM Accuracy: {accuracy_score(y_test, lgbm_preds):.4f}")

### 7.5 Neural Network (PyTorch)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

class SimpleNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

BATCH = 2048
train_ds = TensorDataset(torch.FloatTensor(X_train_sc), torch.FloatTensor(y_train))
test_ds  = TensorDataset(torch.FloatTensor(X_test_sc),  torch.FloatTensor(y_test))
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH)

model = SimpleNet(X_train_sc.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

epochs = 30
best_acc = 0
history = {'train_loss': [], 'val_acc': []}

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)

    model.eval()
    all_preds = []
    with torch.no_grad():
        for xb, yb in test_loader:
            all_preds.append(model(xb.to(device)).cpu())
    val_probs = torch.cat(all_preds).numpy()
    val_acc = accuracy_score(y_test, (val_probs > 0.5).astype(int))
    avg_loss = epoch_loss / len(train_ds)
    scheduler.step(avg_loss)

    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

model.load_state_dict(best_state)
print(f"\nBest validation accuracy: {best_acc:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    nn_probs = model(torch.FloatTensor(X_test_sc).to(device)).cpu().numpy()
nn_preds = (nn_probs > 0.5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], color='steelblue')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')
axes[1].plot(history['val_acc'], color='coral')
axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.show()

## 8. Evaluation & Comparison

### 8.1 Results Summary

In [ ]:
models_eval = [
    ('Logistic Regression', lr_preds, lr_probs),
    ('Random Forest',       rf_preds, rf_probs),
    ('XGBoost',             xgb_preds, xgb_probs),
    ('LightGBM',            lgbm_preds, lgbm_probs),
    ('Neural Network',      nn_preds, nn_probs),
]

rows = []
for name, preds, probs in models_eval:
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'F1 Score': f1_score(y_test, preds),
        'ROC AUC': roc_auc_score(y_test, probs),
    })

results_df = pd.DataFrame(rows).set_index('Model').sort_values('Accuracy', ascending=False)
print(results_df.to_string())

In [ ]:
results_df.plot.barh(figsize=(10, 5), color=['#4a90d9', '#2ecc71', '#e74c3c'])
plt.xlabel('Score')
plt.title('Model Comparison')
plt.xlim(0.5, 1.0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 8.2 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, preds, _) in zip(axes, models_eval):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Low', 'High'], yticklabels=['Low', 'High'])
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

### 8.3 ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))
for name, _, probs in models_eval:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 8.4 Classification Reports

In [ ]:
for name, preds, _ in models_eval:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, preds, target_names=['Low', 'High']))

### 8.5 Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, clf) in zip(axes, [('Random Forest', rf), ('XGBoost', xgb), ('LightGBM', lgbm)]):
    imp = pd.Series(clf.feature_importances_, index=feature_cols).sort_values()
    imp.plot.barh(ax=ax, color='steelblue')
    ax.set_title(f'{name}')
    ax.set_xlabel('Importance')
plt.suptitle('Feature Importance by Model', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Conclusion

In [ ]:
best = results_df['Accuracy'].idxmax()
print(f"Best model: {best}")
print(f"  Accuracy: {results_df.loc[best, 'Accuracy']:.4f}")
print(f"  F1 Score: {results_df.loc[best, 'F1 Score']:.4f}")
print(f"  ROC AUC:  {results_df.loc[best, 'ROC AUC']:.4f}")